# Лабораторная работа 3. 
# Сетевые алгоритмы. Динамические алгоритмы поиска путей.


## Выполнил студент группы Кесслер Алексей БВТ2306
***

### Задание

1.  Реализовать алгоритм поиска кратчайшего расстояния между двумя вершинами ориентированного взвешенного графа в соответствии с вариантом. 

2.  Предусмотреть задание графа в виде матрицы смежности/инцидентности, читаемой из файла, либо графически с помощью пользовательского интерфейса. 

3.  Разработать графический интерфейс пользователя с визуализацией графа и отображением кратчайшего расстояния между задаваемыми пользователем вершинами.

4. По результатам работы проанализировать временную сложность работы заданного алгоритма в зависимости от числа узлов и ребер графа.
Данные представить в виде таблицы.



### Алгоритмы:

Алгоритм Флойда-Уоршелла| Алгоритм Дейкстры | Алгоритм Беллмана-Форда | Алгоритм Джонсона| Алгоритм Левита | Алгоритм Йена



### Выполнение:

In [1]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import sys
import random
import time


class Graph:
    def __init__(self, vertices):
        self.V = vertices
        self.graph = [[0 for _ in range(vertices)] for _ in range(vertices)]

    def johnson(self):
        extended_graph = Graph(self.V + 1)
        extended_graph.graph = [row + [0] for row in self.graph] + [[0 for _ in range(self.V + 1)]]

        potential = extended_graph.bellman_ford(self.V)
        if potential is None:
            return None

        self.reweight(potential[:-1])

        all_shortest_paths = []
        for i in range(self.V):
            distances = self.dijkstra(i)
            for j in range(self.V):
                if distances[j] != sys.maxsize:
                    distances[j] -= potential[i] - potential[j]
            all_shortest_paths.append(distances)

        return all_shortest_paths

    def bellman_ford(self, src):
        dist = [sys.maxsize] * self.V
        dist[src] = 0

        for _ in range(self.V - 1):
            for u in range(self.V):
                for v in range(self.V):
                    if self.graph[u][v] != 0 and dist[u] != sys.maxsize and dist[u] + self.graph[u][v] < dist[v]:
                        dist[v] = dist[u] + self.graph[u][v]

        for u in range(self.V):
            for v in range(self.V):
                if self.graph[u][v] != 0 and dist[u] != sys.maxsize and dist[u] + self.graph[u][v] < dist[v]:
                    return None
        return dist

    def reweight(self, potential):
        for i in range(self.V):
            for j in range(self.V):
                if self.graph[i][j] != 0:
                    self.graph[i][j] += potential[i] - potential[j]

    def dijkstra(self, src):
        dist = [sys.maxsize] * self.V
        dist[src] = 0
        sptSet = [False] * self.V

        for _ in range(self.V):
            min_dist = sys.maxsize
            min_index = -1
            for v in range(self.V):
                if not sptSet[v] and dist[v] < min_dist:
                    min_dist = dist[v]
                    min_index = v

            u = min_index
            sptSet[u] = True

            for v in range(self.V):
                if self.graph[u][v] > 0 and not sptSet[v] and dist[u] + self.graph[u][v] < dist[v]:
                    dist[v] = dist[u] + self.graph[u][v]

        return dist

    def draw_graph(self, figure):
        G = nx.DiGraph()
        for i in range(self.V):
            for j in range(self.V):
                if self.graph[i][j] != 0:
                    G.add_edge(i, j, weight=self.graph[i][j])

        pos = nx.spring_layout(G)
        plt.clf()
        nx.draw(
            G, pos,
            with_labels=True,
            node_size=800,
            node_color="#A3CCE9",
            font_size=12,
            font_color="black",
            edge_color="#7A7A7A",
            arrows=True,
            width=2
        )
        edge_labels = nx.get_edge_attributes(G, 'weight')
        nx.draw_networkx_edge_labels(
            G, pos, edge_labels=edge_labels, font_color="red", font_size=10
        )
        figure.canvas.draw()


class GUI:
    def __init__(self, master):
        self.master = master
        self.master.title("Алгоритм Джонсона: Кратчайший путь")
        self.master.geometry("1080x720")
        self.master.configure(bg="#F0F0F0")

        self.top_frame = tk.Frame(master, bg="#2B547E", pady=10)
        self.top_frame.pack(fill="x")

        header_label = tk.Label(
            self.top_frame,
            text="Алгоритм Джонсона — Визуализация кратчайших путей",
            font=("Verdana", 16),
            bg="#2B547E",
            fg="white"
        )
        header_label.pack()

        self.control_frame = tk.Frame(master, bg="#F0F0F0", width=300)
        self.control_frame.pack(side="right", fill="y")

        self.button_browse = ttk.Button(self.control_frame, text="Загрузить граф из файла", command=self.browse_file)
        self.button_browse.pack(pady=10, padx=10, fill="x")

        source_label = tk.Label(self.control_frame, text="Начальная вершина", font=("Arial", 11), bg="#F0F0F0")
        source_label.pack(pady=(20, 5))
        self.source_entry = ttk.Entry(self.control_frame, width=10)
        self.source_entry.pack(pady=(0, 10), padx=10)

        dest_label = tk.Label(self.control_frame, text="Конечная вершина", font=("Arial", 11), bg="#F0F0F0")
        dest_label.pack(pady=(20, 5))
        self.dest_entry = ttk.Entry(self.control_frame, width=10)
        self.dest_entry.pack(pady=(0, 10), padx=10)

        self.button_calculate = ttk.Button(self.control_frame, text="Вычислить путь", command=self.calculate_shortest_path)
        self.button_calculate.pack(pady=10, padx=10, fill="x")

        self.button_analyze_time = ttk.Button(self.control_frame, text="Анализ времени", command=self.analyze_time_complexity)
        self.button_analyze_time.pack(pady=10, padx=10, fill="x")

        self.canvas_frame = tk.Frame(master, bg="white", bd=2, relief="ridge")
        self.canvas_frame.pack(side="left", fill="both", expand=True, padx=10, pady=10)

        self.figure = plt.figure(figsize=(8, 6))
        self.canvas = FigureCanvasTkAgg(self.figure, self.canvas_frame)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)

        self.adjacency_matrix = None

    def browse_file(self):
        file_path = filedialog.askopenfilename(filetypes=[("Текстовые файлы", "*.txt")])
        if not file_path:
            return

        try:
            self.adjacency_matrix = self.read_adjacency_matrix(file_path)
            self.update_graph()
        except Exception as e:
            messagebox.showerror("Ошибка", f"Ошибка при чтении файла: {e}")

    def update_graph(self):
        if self.adjacency_matrix:
            g = Graph(len(self.adjacency_matrix))
            g.graph = self.adjacency_matrix
            g.draw_graph(self.figure)

    def calculate_shortest_path(self):
        if not self.adjacency_matrix:
            messagebox.showerror("Ошибка", "Загрузите граф (матрицу смежности)!")
            return

        try:
            source = int(self.source_entry.get())
            destination = int(self.dest_entry.get())

            if not (0 <= source < len(self.adjacency_matrix)) or not (0 <= destination < len(self.adjacency_matrix)):
                messagebox.showerror("Ошибка", "Некорректный номер вершины!")
                return

            g = Graph(len(self.adjacency_matrix))
            g.graph = self.adjacency_matrix
            shortest_paths = g.johnson()

            if shortest_paths is None:
                messagebox.showerror("Ошибка", "Граф содержит цикл отрицательного веса.")
                return

            result = shortest_paths[source][destination]
            message = (
                f"Кратчайшее расстояние от вершины {source} до вершины {destination}: {result}"
                if result != sys.maxsize
                else f"Путь недостижим от вершины {source} до {destination}."
            )
            messagebox.showinfo("Кратчайший путь", message)
        except ValueError:
            messagebox.showerror("Ошибка", "Убедитесь, что введены корректные номера вершин!")

    def analyze_time_complexity(self):
        try:
            sizes = [50, 100, 150, 200]
            results = []

            for size in sizes:
                adj_matrix = [[0 for _ in range(size)] for _ in range(size)]
                edges = 0

                for i in range(size):
                    for j in range(size):
                        if i != j and random.random() < 0.3:
                            weight = random.randint(1, 10)
                            adj_matrix[i][j] = weight
                            edges += 1

                g = Graph(size)
                g.graph = adj_matrix

                start_time = time.time()
                g.johnson()
                elapsed_time = time.time() - start_time

                results.append((size, edges, elapsed_time))

            table_window = tk.Toplevel(self.master)
            table_window.title("Анализ временной сложности")

            tree = ttk.Treeview(table_window, columns=("vertices", "edges", "time"), show="headings")
            tree.heading("vertices", text="Количество вершин")
            tree.heading("edges", text="Количество рёбер")
            tree.heading("time", text="Время выполнения (сек.)")
            tree.pack(fill="both", expand=True)

            for vertices, edges, time_taken in results:
                tree.insert("", "end", values=(vertices, edges, f"{time_taken:.6f}"))

        except Exception as e:
            messagebox.showerror("Ошибка", f"Ошибка при анализе времени работы: {e}")

    @staticmethod
    def read_adjacency_matrix(file_path):
        with open(file_path, "r") as file:
            return [[int(num) for num in line.split()] for line in file]


if __name__ == "__main__":
    root = tk.Tk()
    app = GUI(root)
    root.mainloop()


2025-05-21 17:54:53.182 Python[98698:4072723] +[IMKClient subclass]: chose IMKClient_Modern
2025-05-21 17:54:53.183 Python[98698:4072723] +[IMKInputSession subclass]: chose IMKInputSession_Modern


<Figure size 844x651 with 0 Axes>

### Вывод